In [1]:
# 训练Transformer模型

import torch
from torch.utils.data import Dataset, DataLoader
from model import Word2Idx

# ====================== 固定配置 ======================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VOCAB_SIZE = 8000          # 分词模型词汇表
MAX_LEN = 512             # 最大序列长度
BATCH_SIZE = 2            # 手写模型用小批次
EPOCHS = 10               # 训练轮数
D_MODEL = 512             # Transformer嵌入维度（和你模型一致）

# 初始化中英分词器
zh_tokenizer = Word2Idx()
en_tokenizer = Word2Idx()

# 加载模型
zh_tokenizer.load_model("../data/Chinese_English_parallel_corpus/Chinese.model")
en_tokenizer.load_model("../data/Chinese_English_parallel_corpus/English.model")


✅ 模型加载成功，词汇表大小: 8000
✅ 模型加载成功，词汇表大小: 8000


In [2]:
import os
from typing import Tuple
class TranslationDataset(Dataset):
    def __init__(self, zh_path: str, en_path: str,
                 zh_tokenizer, en_tokenizer,
                 max_len: int = 512,
                 device=None):
        """
        翻译数据集

        Args:
            zh_path: 中文文件路径
            en_path: 英文文件路径
            zh_tokenizer: 中文tokenizer
            en_tokenizer: 英文tokenizer
            max_len: 最大序列长度
            device: 设备（建议在collate_fn中处理，而不是这里）
        """
        # 读取平行语料
        self.zh_sentences, self.en_sentences = self._load_parallel_corpus(zh_path, en_path)

        # 验证数据对齐
        assert len(self.zh_sentences) == len(self.en_sentences), \
            f"中英文句子数量不匹配: {len(self.zh_sentences)} vs {len(self.en_sentences)}"

        self.zh_tokenizer = zh_tokenizer
        self.en_tokenizer = en_tokenizer
        self.max_len = max_len
        self.device = device  # 可选，建议在外部移动tensor到device

    def _load_parallel_corpus(self, zh_path: str, en_path: str) -> Tuple[list, list]:
        """加载平行语料，带错误处理"""
        if not os.path.exists(zh_path):
            raise FileNotFoundError(f"中文文件不存在: {zh_path}")
        if not os.path.exists(en_path):
            raise FileNotFoundError(f"英文文件不存在: {en_path}")

        # 读取文件，过滤空行
        with open(zh_path, 'r', encoding='utf-8') as f:
            zh_sentences = [line.strip() for line in f if line.strip()]

        with open(en_path, 'r', encoding='utf-8') as f:
            en_sentences = [line.strip() for line in f if line.strip()]

        # 确保数量一致（取较短的）
        min_len = min(len(zh_sentences), len(en_sentences))
        if len(zh_sentences) != len(en_sentences):
            print(f"⚠️ 警告: 中英文数量不一致，将截取到 {min_len} 对")
            zh_sentences = zh_sentences[:min_len]
            en_sentences = en_sentences[:min_len]

        return zh_sentences, en_sentences

    def __len__(self) -> int:
        return len(self.zh_sentences)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        返回:
            src: 源语言（中文）输入 [seq_len]
            tgt_input: 目标语言（英文）解码器输入 [seq_len]
            tgt_label: 目标语言标签 [seq_len]
        """
        # 获取句子对
        zh_text = self.zh_sentences[idx]
        en_text = self.en_sentences[idx]

        # 编码（添加BOS和EOS）
        src = self.zh_tokenizer.text_to_ids(
            zh_text,
            max_len=self.max_len,
            add_bos=True,   # 添加<BOS>
            add_eos=True    # 添加<EOS>
        )

        tgt = self.en_tokenizer.text_to_ids(
            en_text,
            max_len=self.max_len,
            add_bos=True,
            add_eos=True
        )

        # 创建解码器输入和标签
        # 标准做法: tgt_input = [BOS, token1, token2, ..., tokenN]
        #          tgt_label = [token1, token2, ..., tokenN, EOS]
        tgt_input = tgt[:-1]  # 去掉最后一个EOS
        tgt_label = tgt[1:]   # 去掉第一个BOS

        # 验证长度一致性
        assert len(tgt_input) == len(tgt_label), \
            f"tgt_input和tgt_label长度不一致: {len(tgt_input)} vs {len(tgt_label)}"

        # 转换为tensor（不在这里移动到device，保持CPU）
        return (
            torch.tensor(src, dtype=torch.long),
            torch.tensor(tgt_input, dtype=torch.long),
            torch.tensor(tgt_label, dtype=torch.long)
        )


def collate_fn(batch, pad_id: int = 0):
    """
    自定义批处理函数，处理变长序列

    Args:
        batch: 单个样本列表 [(src, tgt_input, tgt_label), ...]
        pad_id: 填充ID

    Returns:
        填充后的batch tensors
    """
    src_batch, tgt_input_batch, tgt_label_batch = zip(*batch)

    # 获取batch中的最大长度
    src_max_len = max(seq.size(0) for seq in src_batch)
    tgt_max_len = max(seq.size(0) for seq in tgt_input_batch)

    # 填充函数
    def pad_sequences(sequences, max_len):
        padded = torch.stack([
            torch.nn.functional.pad(seq, (0, max_len - seq.size(0)), value=pad_id)
            for seq in sequences
        ])
        return padded

    # 填充所有序列
    src_padded = pad_sequences(src_batch, src_max_len)
    tgt_input_padded = pad_sequences(tgt_input_batch, tgt_max_len)
    tgt_label_padded = pad_sequences(tgt_label_batch, tgt_max_len)

    # 创建attention mask（可选）
    src_mask = (src_padded != pad_id)
    tgt_mask = (tgt_input_padded != pad_id)

    return {
        'src': src_padded,
        'tgt_input': tgt_input_padded,
        'tgt_label': tgt_label_padded,
        'src_mask': src_mask,
        'tgt_mask': tgt_mask,
        'src_lengths': torch.tensor([seq.size(0) for seq in src_batch]),
        'tgt_lengths': torch.tensor([seq.size(0) for seq in tgt_input_batch])
    }

In [3]:
# 这里导入你自己写的 Transformer 模型！
import torch
import torch.nn as nn
from tqdm import tqdm
from model import Transformer

# 配置参数
VOCAB_SIZE = 8000
D_MODEL = 512
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dataset = TranslationDataset(
    zh_path='../data/Chinese_English_parallel_corpus/chinese.txt',
    en_path='../data/Chinese_English_parallel_corpus/english.txt',
    max_len=MAX_LEN,
    zh_tokenizer=zh_tokenizer,
    en_tokenizer=en_tokenizer
)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

# 初始化模型（mask在模型内部处理，不需要外部传入）
model = Transformer(
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    max_len=MAX_LEN,
    pad_token_id=0  # 确保传入pad_token_id
).to(DEVICE)

# 损失函数（忽略padding）
criterion = nn.CrossEntropyLoss(ignore_index=0)

# 优化器
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print(f"🚀 开始训练，设备: {DEVICE}")
print(f"训练样本数: {len(dataset)}")
print("-" * 50)

# 训练循环
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    # 使用进度条
    loop = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch in loop:  # batch是一个字典
        # 从字典中取出数据
        src = batch['src'].to(DEVICE)
        tgt_input = batch['tgt_input'].to(DEVICE)
        tgt_label = batch['tgt_label'].to(DEVICE)

        # 前向传播（mask在模型内部自动创建）
        optimizer.zero_grad()
        output = model(src, tgt_input)  # 不需要传入mask！

        # 计算损失
        output = output.reshape(-1, VOCAB_SIZE)
        tgt_label = tgt_label.reshape(-1)
        loss = criterion(output, tgt_label)

        # 反向传播
        loss.backward()
        optimizer.step()

        # 记录损失
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    # 打印epoch结果
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1} | 平均损失: {avg_loss:.4f} | 困惑度: {torch.exp(torch.tensor(avg_loss)):.2f}")
    print("-" * 50)

# 保存模型
# torch.save(model.state_dict(), "translation_model.pth")
# print("✅ 模型训练完成，已保存为 translation_model.pth")

🚀 开始训练，设备: cpu
训练样本数: 100000
--------------------------------------------------


Epoch 1/10:  11%|█         | 336/3125 [41:37<5:45:31,  7.43s/it, loss=7.11]


KeyboardInterrupt: 

In [ ]:
def translate(text):
    model.eval()
    with torch.no_grad():
        # 中文→索引
        src = zh_tokenizer.text_to_ids(text, MAX_LEN)
        src = torch.tensor([src], dtype=torch.long).to(DEVICE)

        # 初始化解码器输入（以<BOS>开头）
        tgt_input = torch.tensor([[2]], dtype=torch.long).to(DEVICE)

        # 生成翻译
        for _ in range(MAX_LEN):
            output = model(src, tgt_input)
            # 取最后一个词的预测结果
            next_token = output.argmax(-1)[:, -1]
            tgt_input = torch.cat([tgt_input, next_token.unsqueeze(0)], dim=1)

            # 遇到<EOS>停止
            if next_token.item() == 3:
                break

        # 索引→英文
        result = en_tokenizer.ids_to_text(tgt_input.squeeze().tolist())
        return result


# 测试翻译
print(translate("我爱机器学习"))